In [1]:
import os
import sys

sys.path.append(os.path.abspath("../src"))

In [2]:
import pandas as pd
import numpy as np
import joblib
import optuna

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from models import get_models

from evaluation import (
    evaluate_model,
    evaluate_models,
    compare_models
)

from tuning import (
    grid_search_tune,
    random_search_tune,
    optuna_tune,
    cross_validate
)

from stacking import CardiacX

In [3]:
X_train = pd.read_csv("../data/processed/X_train_scaled.csv")
X_test = pd.read_csv("../data/processed/X_test_scaled.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

In [4]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (800, 12)
X_test : (200, 12)
y_train: (800,)
y_test : (200,)


In [5]:
models = get_models()

list(models.keys())

['Logistic Regression',
 'Decision Tree',
 'Random Forest',
 'KNN',
 'SVM',
 'XGBoost']

In [6]:
for model in models.values():
    model.fit(X_train, y_train)

/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [14:54:11] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [7]:
baseline_results = evaluate_models(
    models,
    X_test,
    y_test
)

baseline_results.sort_values(
    by="Accuracy",
    ascending=False
)

,Accuracy,Precision,Recall,F1 Score,AUC,Hamming Loss,Jaccard Score,Log Loss,MCC
Random Forest,0.985,0.982906,0.991379,0.987124,0.998974,0.015,0.974576,0.077212,0.969212
Logistic Regression,0.980,0.974576,0.991379,0.982906,0.998255,0.020,0.966387,0.068578,0.959017
XGBoost,0.980,0.982759,0.982759,0.982759,0.998871,0.020,0.966102,0.040606,0.958949
Decision Tree,0.975,0.991150,0.965517,0.978166,0.976806,0.025,0.957265,0.901091,0.949384
SVM,0.970,0.958333,0.991379,0.974576,0.997742,0.030,0.950413,0.070063,0.938818
KNN,0.915,0.923077,0.931034,0.927039,0.973573,0.085,0.864000,0.497690,0.825290


In [8]:
RF_GRID = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

RF_RANDOM = {
    "n_estimators": [100, 200, 300, 400],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "bootstrap": [True, False]
}

In [9]:
SVM_GRID = {
    "C": [0.1, 1, 10, 100],
    "kernel": ["linear", "rbf", "poly"],
    "gamma": ["scale", "auto"]
}

SVM_RANDOM = {
    "C": [0.01, 0.1, 1, 10, 100],
    "kernel": ["linear", "rbf", "poly", "sigmoid"],
    "gamma": ["scale", "auto"],
    "degree": [2, 3, 4]
}

In [10]:
DT_GRID = {
    "criterion": ["gini", "entropy"],
    "max_depth": [None, 5, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

DT_RANDOM = {
    "criterion": ["gini", "entropy"],
    "max_depth": [None, 5, 10, 20, 30],
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf": [1, 2, 4, 6]
}

In [11]:
LR_GRID = {
    "C": [0.01, 0.1, 1, 10, 100],
    "solver": ["lbfgs", "liblinear"],
    "max_iter": [100, 200, 500]
}

LR_RANDOM = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "solver": ["lbfgs", "liblinear"],
    "max_iter": [100, 200, 500, 1000]
}

In [12]:
KNN_GRID = {
    "n_neighbors": [3, 5, 7, 9, 11],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}

KNN_RANDOM = {
    "n_neighbors": [1, 3, 5, 7, 9, 11, 13, 15],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan", "minkowski"]
}

In [13]:
XGB_GRID = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2],
    "subsample": [0.8, 1]
}

XGB_RANDOM = {
    "n_estimators": [100, 200, 300, 400],
    "max_depth": [3, 5, 7, 9],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1]
}

In [14]:
grid_configs = {
    "Random Forest": RF_GRID,
    "SVM": SVM_GRID,
    "Decision Tree": DT_GRID,
    "Logistic Regression": LR_GRID,
    "KNN": KNN_GRID,
    "XGBoost": XGB_GRID
}

In [15]:
random_configs = {
    "Random Forest": RF_RANDOM,
    "SVM": SVM_RANDOM,
    "Decision Tree": DT_RANDOM,
    "Logistic Regression": LR_RANDOM,
    "KNN": KNN_RANDOM,
    "XGBoost": XGB_RANDOM
}

In [16]:
grid_models = {}
grid_results = {}

for name, params in grid_configs.items():

    print(f"\n{'='*60}")
    print(f"Grid Search : {name}")
    print(f"{'='*60}")

    result = grid_search_tune(
        model=models[name],
        param_grid=params,
        X_train=X_train,
        y_train=y_train
    )

    grid_models[name] = result["best_model"]

    grid_results[name] = {
        "Best Score": result["best_score"],
        "Best Params": result["best_params"]
    }

    print(result["best_params"])


Grid Search : Random Forest


{'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}

Grid Search : SVM


/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/

{'C': 100, 'gamma': 'scale', 'kernel': 'linear'}

Grid Search : Decision Tree
{'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 10}

Grid Search : Logistic Regression
{'C': 10, 'max_iter': 100, 'solver': 'lbfgs'}

Grid Search : KNN
{'metric': 'manhattan', 'n_neighbors': 9, 'weights': 'distance'}

Grid Search : XGBoost


/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [14:54:17] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [14:54:17] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [14:54:17] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [14:54:17] WARNING: /Users/runn

{'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 100, 'subsample': 0.8}


In [17]:
grid_summary = pd.DataFrame(grid_results).T

grid_summary

,Best Score,Best Params
Random Forest,0.97125,"{'max_depth': None, 'min_samples_leaf': 1, 'mi..."
SVM,0.965,"{'C': 100, 'gamma': 'scale', 'kernel': 'linear'}"
Decision Tree,0.9575,"{'criterion': 'gini', 'max_depth': None, 'min_..."
Logistic Regression,0.96875,"{'C': 10, 'max_iter': 100, 'solver': 'lbfgs'}"
KNN,0.9625,"{'metric': 'manhattan', 'n_neighbors': 9, 'wei..."
XGBoost,0.975,"{'learning_rate': 0.1, 'max_depth': 7, 'n_esti..."


In [18]:
grid_metrics = evaluate_models(
    grid_models,
    X_test,
    y_test
)

grid_metrics.sort_values(
    by="Accuracy",
    ascending=False
)

,Accuracy,Precision,Recall,F1 Score,AUC,Hamming Loss,Jaccard Score,Log Loss,MCC
Decision Tree,0.990,0.991379,0.991379,0.991379,0.993432,0.010,0.982906,0.202798,0.979475
XGBoost,0.990,0.991379,0.991379,0.991379,0.999076,0.010,0.982906,0.039473,0.979475
Random Forest,0.985,0.982906,0.991379,0.987124,0.998974,0.015,0.974576,0.077212,0.969212
SVM,0.985,0.982906,0.991379,0.987124,0.998358,0.015,0.974576,0.062921,0.969212
Logistic Regression,0.980,0.982759,0.982759,0.982759,0.998666,0.020,0.966102,0.054078,0.958949
KNN,0.950,0.964912,0.948276,0.956522,0.992713,0.050,0.916667,0.113525,0.897897


In [19]:
random_models = {}
random_results = {}

for name, params in random_configs.items():

    print(f"\n{'='*60}")
    print(f"Random Search : {name}")
    print(f"{'='*60}")

    result = random_search_tune(
        model=models[name],
        param_distributions=params,
        X_train=X_train,
        y_train=y_train,
        n_iter=20
    )

    random_models[name] = result["best_model"]

    random_results[name] = {
        "Best Score": result["best_score"],
        "Best Params": result["best_params"]
    }

    print(result["best_params"])


Random Search : Random Forest
{'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 20, 'bootstrap': False}

Random Search : SVM


/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/

{'kernel': 'linear', 'gamma': 'scale', 'degree': 3, 'C': 100}

Random Search : Decision Tree
{'min_samples_split': 15, 'min_samples_leaf': 2, 'max_depth': 20, 'criterion': 'gini'}

Random Search : Logistic Regression
{'solver': 'liblinear', 'max_iter': 500, 'C': 10}

Random Search : KNN
{'weights': 'distance', 'n_neighbors': 9, 'metric': 'manhattan'}

Random Search : XGBoost


/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [14:54:24] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [14:54:24] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [14:54:24] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [14:54:24] WARNING: /Users/runn

{'subsample': 0.8, 'n_estimators': 300, 'max_depth': 9, 'learning_rate': 0.1}


In [20]:
random_summary = pd.DataFrame(random_results).T

random_summary

,Best Score,Best Params
Random Forest,0.97875,"{'n_estimators': 100, 'min_samples_split': 2, ..."
SVM,0.965,"{'kernel': 'linear', 'gamma': 'scale', 'degree..."
Decision Tree,0.95125,"{'min_samples_split': 15, 'min_samples_leaf': ..."
Logistic Regression,0.96875,"{'solver': 'liblinear', 'max_iter': 500, 'C': 10}"
KNN,0.9625,"{'weights': 'distance', 'n_neighbors': 9, 'met..."
XGBoost,0.97375,"{'subsample': 0.8, 'n_estimators': 300, 'max_d..."


In [21]:
random_metrics = evaluate_models(
    random_models,
    X_test,
    y_test
)

random_metrics.sort_values(
    by="Accuracy",
    ascending=False
)

,Accuracy,Precision,Recall,F1 Score,AUC,Hamming Loss,Jaccard Score,Log Loss,MCC
Random Forest,0.995,1.000000,0.991379,0.995671,0.999436,0.005,0.991379,0.058751,0.989806
XGBoost,0.990,0.991379,0.991379,0.991379,0.999282,0.010,0.982906,0.038299,0.979475
SVM,0.985,0.982906,0.991379,0.987124,0.998358,0.015,0.974576,0.064970,0.969212
Logistic Regression,0.980,0.982759,0.982759,0.982759,0.998666,0.020,0.966102,0.054186,0.958949
Decision Tree,0.975,0.966387,0.991379,0.978723,0.992970,0.025,0.958333,0.220216,0.948887
KNN,0.950,0.964912,0.948276,0.956522,0.992713,0.050,0.916667,0.113525,0.897897


In [22]:
def rf_objective(trial):

    model = RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 400),
        max_depth=trial.suggest_int("max_depth", 5, 50),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 10),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 5),
        random_state=42
    )

    return cross_validate(
        model,
        X_train,
        y_train
    )["mean"]

In [23]:
def svm_objective(trial):

    params = {
        "C": trial.suggest_float("C", 0.01, 100, log=True),
        "kernel": trial.suggest_categorical(
            "kernel",
            ["linear", "rbf", "poly"]
        ),
        "gamma": trial.suggest_categorical(
            "gamma",
            ["scale", "auto"]
        ),
        "probability": True
    }

    if params["kernel"] == "poly":
        params["degree"] = trial.suggest_int(
            "degree",
            2,
            4
        )

    model = SVC(**params)

    return cross_validate(
        model,
        X_train,
        y_train
    )["mean"]

In [24]:
def dt_objective(trial):

    model = DecisionTreeClassifier(
        criterion=trial.suggest_categorical(
            "criterion",
            ["gini", "entropy"]
        ),
        max_depth=trial.suggest_int(
            "max_depth",
            3,
            30
        ),
        min_samples_split=trial.suggest_int(
            "min_samples_split",
            2,
            15
        ),
        min_samples_leaf=trial.suggest_int(
            "min_samples_leaf",
            1,
            6
        ),
        random_state=42
    )

    return cross_validate(
        model,
        X_train,
        y_train
    )["mean"]

In [25]:
def lr_objective(trial):

    model = LogisticRegression(
        C=trial.suggest_float(
            "C",
            0.001,
            100,
            log=True
        ),
        solver=trial.suggest_categorical(
            "solver",
            ["lbfgs", "liblinear"]
        ),
        max_iter=500
    )

    return cross_validate(
        model,
        X_train,
        y_train
    )["mean"]

In [26]:
def knn_objective(trial):

    model = KNeighborsClassifier(
        n_neighbors=trial.suggest_int(
            "n_neighbors",
            1,
            20
        ),
        weights=trial.suggest_categorical(
            "weights",
            ["uniform", "distance"]
        ),
        metric=trial.suggest_categorical(
            "metric",
            ["euclidean", "manhattan", "minkowski"]
        )
    )

    return cross_validate(
        model,
        X_train,
        y_train
    )["mean"]

In [27]:
def xgb_objective(trial):

    model = XGBClassifier(
        n_estimators=trial.suggest_int(
            "n_estimators",
            100,
            400
        ),
        max_depth=trial.suggest_int(
            "max_depth",
            3,
            10
        ),
        learning_rate=trial.suggest_float(
            "learning_rate",
            0.01,
            0.2
        ),
        subsample=trial.suggest_float(
            "subsample",
            0.7,
            1.0
        ),
        eval_metric="logloss"
    )

    return cross_validate(
        model,
        X_train,
        y_train
    )["mean"]

In [28]:
optuna_objectives = {
    "Random Forest": rf_objective,
    "SVM": svm_objective,
    "Decision Tree": dt_objective,
    "Logistic Regression": lr_objective,
    "KNN": knn_objective,
    "XGBoost": xgb_objective
}

In [29]:
optuna_studies = {}

for name, objective in optuna_objectives.items():

    print(f"\n{'='*60}")
    print(f"Optuna : {name}")
    print(f"{'='*60}")

    study = optuna_tune(
        objective=objective,
        n_trials=30
    )

    optuna_studies[name] = study

    print(study.best_params)

[I 2026-07-05 14:54:24,930] A new study created in memory with name: no-name-7e5fb8d9-37d3-4c45-874f-edc84026f8b7



Optuna : Random Forest


[I 2026-07-05 14:54:25,105] Trial 0 finished with value: 0.9574999999999999 and parameters: {'n_estimators': 173, 'max_depth': 30, 'min_samples_split': 6, 'min_samples_leaf': 4}. Best is trial 0 with value: 0.9574999999999999.
[I 2026-07-05 14:54:25,423] Trial 1 finished with value: 0.95625 and parameters: {'n_estimators': 250, 'max_depth': 25, 'min_samples_split': 6, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.9574999999999999.
[I 2026-07-05 14:54:25,766] Trial 2 finished with value: 0.9650000000000001 and parameters: {'n_estimators': 263, 'max_depth': 38, 'min_samples_split': 8, 'min_samples_leaf': 3}. Best is trial 2 with value: 0.9650000000000001.
[I 2026-07-05 14:54:25,964] Trial 3 finished with value: 0.9625 and parameters: {'n_estimators': 194, 'max_depth': 49, 'min_samples_split': 9, 'min_samples_leaf': 3}. Best is trial 2 with value: 0.9650000000000001.
[I 2026-07-05 14:54:26,275] Trial 4 finished with value: 0.9550000000000001 and parameters: {'n_estimators': 348, '

{'n_estimators': 318, 'max_depth': 50, 'min_samples_split': 3, 'min_samples_leaf': 1}

Optuna : SVM


[I 2026-07-05 14:54:33,030] Trial 5 finished with value: 0.8575000000000002 and parameters: {'C': 1.1643420392860675, 'kernel': 'poly', 'gamma': 'scale', 'degree': 4}. Best is trial 4 with value: 0.96.
/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be rem

...

{'C': 13.785793982263556, 'solver': 'lbfgs'}

Optuna : KNN


[I 2026-07-05 14:54:35,320] Trial 14 finished with value: 0.96125 and parameters: {'n_neighbors': 8, 'weights': 'distance', 'metric': 'manhattan'}. Best is trial 14 with value: 0.96125.
[I 2026-07-05 14:54:35,335] Trial 15 finished with value: 0.9625 and parameters: {'n_neighbors': 9, 'weights': 'distance', 'metric': 'manhattan'}. Best is trial 15 with value: 0.9625.
[I 2026-07-05 14:54:35,351] Trial 16 finished with value: 0.9637499999999999 and parameters: {'n_neighbors': 10, 'weights': 'distance', 'metric': 'manhattan'}. Best is trial 16 with value: 0.9637499999999999.
[I 2026-07-05 14:54:35,366] Trial 17 finished with value: 0.9625 and parameters: {'n_neighbors': 9, 'weights': 'distance', 'metric': 'manhattan'}. Best is trial 16 with value: 0.9637499999999999.
[I 2026-07-05 14:54:35,381] Trial 18 finished with value: 0.96125 and parameters: {'n_neighbors': 14, 'weights': 'distance', 'metric': 'manhattan'}. Best is trial 16 with value: 0.9637499999999999.
[I 2026-07-05 14:54:35,397]

{'n_neighbors': 10, 'weights': 'distance', 'metric': 'manhattan'}

Optuna : XGBoost


[I 2026-07-05 14:54:35,772] Trial 4 finished with value: 0.9675 and parameters: {'n_estimators': 279, 'max_depth': 8, 'learning_rate': 0.031129783919123258, 'subsample': 0.7079386138407102}. Best is trial 3 with value: 0.9712500000000001.
[I 2026-07-05 14:54:35,811] Trial 5 finished with value: 0.9700000000000001 and parameters: {'n_estimators': 208, 'max_depth': 7, 'learning_rate': 0.036786462191020955, 'subsample': 0.8064148986390807}. Best is trial 3 with value: 0.9712500000000001.
[I 2026-07-05 14:54:35,851] Trial 6 finished with value: 0.9737500000000001 and parameters: {'n_estimators': 194, 'max_depth': 5, 'learning_rate': 0.08201204574465422, 'subsample': 0.8804376173332585}. Best is trial 6 with value: 0.9737500000000001.
[I 2026-07-05 14:54:35,891] Trial 7 finished with value: 0.9762500000000001 and parameters: {'n_estimators': 276, 'max_depth': 6, 'learning_rate': 0.1665953649751505, 'subsample': 0.9041764382285352}. Best is trial 7 with value: 0.9762500000000001.
[I 2026-07-

{'n_estimators': 276, 'max_depth': 6, 'learning_rate': 0.1665953649751505, 'subsample': 0.9041764382285352}


In [30]:
# Build Optuna-tuned models dynamically from studies
optuna_models = {}
for name, study in optuna_studies.items():

    params = study.best_params

    if name == "Random Forest":

        model = RandomForestClassifier(
            **params,
            random_state=42
        )

    elif name == "SVM":

        model = SVC(
            **params,
            probability=True
        )

    elif name == "Decision Tree":

        model = DecisionTreeClassifier(
            **params,
            random_state=42
        )

    elif name == "Logistic Regression":

        model = LogisticRegression(
            **params,
            max_iter=500
        )

    elif name == "KNN":

        model = KNeighborsClassifier(
            **params
        )

    else:

        model = XGBClassifier(
            **params,
            eval_metric="logloss"
        )

    model.fit(
        X_train,
        y_train
    )

    optuna_models[name] = model

/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [31]:
optuna_metrics = evaluate_models(
    optuna_models,
    X_test,
    y_test
)

optuna_metrics.sort_values(
    by="Accuracy",
    ascending=False
)

,Accuracy,Precision,Recall,F1 Score,AUC,Hamming Loss,Jaccard Score,Log Loss,MCC
Decision Tree,0.990,0.991379,0.991379,0.991379,0.993278,0.010,0.982906,0.211573,0.979475
Random Forest,0.985,0.982906,0.991379,0.987124,0.999179,0.015,0.974576,0.078183,0.969212
SVM,0.980,0.974576,0.991379,0.982906,0.998255,0.020,0.966387,0.067653,0.959017
Logistic Regression,0.980,0.982759,0.982759,0.982759,0.998768,0.020,0.966102,0.053437,0.958949
XGBoost,0.980,0.982759,0.982759,0.982759,0.998974,0.020,0.966102,0.044009,0.958949
KNN,0.960,0.973684,0.956897,0.965217,0.992098,0.040,0.932773,0.120580,0.918359


In [32]:
grid_acc = grid_metrics["Accuracy"].to_dict()
random_acc = random_metrics["Accuracy"].to_dict()
optuna_acc = optuna_metrics["Accuracy"].to_dict()

In [33]:
best_models = {}
best_methods = {}

for model_name in models.keys():

    scores = {
        "Grid Search": grid_acc[model_name],
        "Random Search": random_acc[model_name],
        "Optuna": optuna_acc[model_name]
    }

    best_method = max(
        scores,
        key=scores.get
    )

    best_methods[model_name] = best_method

    if best_method == "Grid Search":
        best_models[model_name] = grid_models[model_name]

    elif best_method == "Random Search":
        best_models[model_name] = random_models[model_name]

    else:
        best_models[model_name] = optuna_models[model_name]

In [34]:
best_method_df = pd.DataFrame({
    "Algorithm": best_methods.keys(),
    "Selected Method": best_methods.values()
})

best_method_df

,Algorithm,Selected Method
0,Logistic Regression,Grid Search
1,Decision Tree,Grid Search
2,Random Forest,Random Search
3,KNN,Optuna
4,SVM,Grid Search
5,XGBoost,Grid Search


In [35]:
final_metrics = evaluate_models(
    best_models,
    X_test,
    y_test
)

final_metrics

,Accuracy,Precision,Recall,F1 Score,AUC,Hamming Loss,Jaccard Score,Log Loss,MCC
Logistic Regression,0.980,0.982759,0.982759,0.982759,0.998666,0.020,0.966102,0.054078,0.958949
Decision Tree,0.990,0.991379,0.991379,0.991379,0.993432,0.010,0.982906,0.202798,0.979475
Random Forest,0.995,1.000000,0.991379,0.995671,0.999436,0.005,0.991379,0.058751,0.989806
KNN,0.960,0.973684,0.956897,0.965217,0.992098,0.040,0.932773,0.120580,0.918359
SVM,0.985,0.982906,0.991379,0.987124,0.998358,0.015,0.974576,0.062921,0.969212
XGBoost,0.990,0.991379,0.991379,0.991379,0.999076,0.010,0.982906,0.039473,0.979475


In [36]:
final_metrics["Tuning Method"] = (
    pd.Series(best_methods)
)

In [37]:
selected_params = {}

for name, model in best_models.items():
    selected_params[name] = model.get_params()

In [38]:
final_metrics["Selected Hyperparameters"] = (
    pd.Series(selected_params)
)

final_metrics

,Accuracy,Precision,Recall,F1 Score,AUC,Hamming Loss,Jaccard Score,Log Loss,MCC,Tuning Method,Selected Hyperparameters
Logistic Regression,0.980,0.982759,0.982759,0.982759,0.998666,0.020,0.966102,0.054078,0.958949,Grid Search,"{'C': 10, 'class_weight': None, 'dual': False,..."
Decision Tree,0.990,0.991379,0.991379,0.991379,0.993432,0.010,0.982906,0.202798,0.979475,Grid Search,"{'ccp_alpha': 0.0, 'class_weight': None, 'crit..."
Random Forest,0.995,1.000000,0.991379,0.995671,0.999436,0.005,0.991379,0.058751,0.989806,Random Search,"{'bootstrap': False, 'ccp_alpha': 0.0, 'class_..."
KNN,0.960,0.973684,0.956897,0.965217,0.992098,0.040,0.932773,0.120580,0.918359,Optuna,"{'algorithm': 'auto', 'leaf_size': 30, 'metric..."
SVM,0.985,0.982906,0.991379,0.987124,0.998358,0.015,0.974576,0.062921,0.969212,Grid Search,"{'C': 100, 'break_ties': False, 'cache_size': ..."
XGBoost,0.990,0.991379,0.991379,0.991379,0.999076,0.010,0.982906,0.039473,0.979475,Grid Search,"{'objective': 'binary:logistic', 'base_score':..."


In [39]:
final_metrics.sort_values(
    by="Accuracy",
    ascending=False
)

,Accuracy,Precision,Recall,F1 Score,AUC,Hamming Loss,Jaccard Score,Log Loss,MCC,Tuning Method,Selected Hyperparameters
Random Forest,0.995,1.000000,0.991379,0.995671,0.999436,0.005,0.991379,0.058751,0.989806,Random Search,"{'bootstrap': False, 'ccp_alpha': 0.0, 'class_..."
Decision Tree,0.990,0.991379,0.991379,0.991379,0.993432,0.010,0.982906,0.202798,0.979475,Grid Search,"{'ccp_alpha': 0.0, 'class_weight': None, 'crit..."
XGBoost,0.990,0.991379,0.991379,0.991379,0.999076,0.010,0.982906,0.039473,0.979475,Grid Search,"{'objective': 'binary:logistic', 'base_score':..."
SVM,0.985,0.982906,0.991379,0.987124,0.998358,0.015,0.974576,0.062921,0.969212,Grid Search,"{'C': 100, 'break_ties': False, 'cache_size': ..."
Logistic Regression,0.980,0.982759,0.982759,0.982759,0.998666,0.020,0.966102,0.054078,0.958949,Grid Search,"{'C': 10, 'class_weight': None, 'dual': False,..."
KNN,0.960,0.973684,0.956897,0.965217,0.992098,0.040,0.932773,0.120580,0.918359,Optuna,"{'algorithm': 'auto', 'leaf_size': 30, 'metric..."


In [40]:
estimators = [
    ("rf", best_models["Random Forest"]),
    ("svm", best_models["SVM"]),
    ("dt", best_models["Decision Tree"]),
    ("lr", best_models["Logistic Regression"]),
    ("knn", best_models["KNN"]),
    ("xgb", best_models["XGBoost"])
]

In [41]:
cardiacx = CardiacX(
    estimators=estimators
)

In [42]:
cardiacx.fit(
    X_train,
    y_train
)

/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [14:54:37] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/vamshi/Documents/ds_proj_cardiacx/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:23

In [43]:
cardiacx_metrics = evaluate_model(
    cardiacx,
    X_test,
    y_test
)

cardiacx_metrics

{'Accuracy': 0.985,
 'Precision': 0.9829059829059829,
 'Recall': 0.9913793103448276,
 'F1 Score': 0.9871244635193133,
 'AUC': 0.9997947454844006,
 'Hamming Loss': 0.015,
 'Jaccard Score': 0.9745762711864406,
 'Log Loss': 0.036366343955560376,
 'MCC': 0.9692123940380459}

In [44]:
cardiacx_df = pd.DataFrame(
    cardiacx_metrics,
    index=["CARDIACX"]
)

cardiacx_df

,Accuracy,Precision,Recall,F1 Score,AUC,Hamming Loss,Jaccard Score,Log Loss,MCC
CARDIACX,0.985,0.982906,0.991379,0.987124,0.999795,0.015,0.974576,0.036366,0.969212


In [45]:
all_models = best_models.copy()

all_models["CARDIACX"] = cardiacx

In [46]:
final_comparison = evaluate_models(
    all_models,
    X_test,
    y_test
)

final_comparison

,Accuracy,Precision,Recall,F1 Score,AUC,Hamming Loss,Jaccard Score,Log Loss,MCC
Logistic Regression,0.980,0.982759,0.982759,0.982759,0.998666,0.020,0.966102,0.054078,0.958949
Decision Tree,0.990,0.991379,0.991379,0.991379,0.993432,0.010,0.982906,0.202798,0.979475
Random Forest,0.995,1.000000,0.991379,0.995671,0.999436,0.005,0.991379,0.058751,0.989806
KNN,0.960,0.973684,0.956897,0.965217,0.992098,0.040,0.932773,0.120580,0.918359
SVM,0.985,0.982906,0.991379,0.987124,0.998358,0.015,0.974576,0.062921,0.969212
XGBoost,0.990,0.991379,0.991379,0.991379,0.999076,0.010,0.982906,0.039473,0.979475
CARDIACX,0.985,0.982906,0.991379,0.987124,0.999795,0.015,0.974576,0.036366,0.969212


In [47]:
final_comparison.sort_values(
    by="Accuracy",
    ascending=False
)

,Accuracy,Precision,Recall,F1 Score,AUC,Hamming Loss,Jaccard Score,Log Loss,MCC
Random Forest,0.995,1.000000,0.991379,0.995671,0.999436,0.005,0.991379,0.058751,0.989806
Decision Tree,0.990,0.991379,0.991379,0.991379,0.993432,0.010,0.982906,0.202798,0.979475
XGBoost,0.990,0.991379,0.991379,0.991379,0.999076,0.010,0.982906,0.039473,0.979475
SVM,0.985,0.982906,0.991379,0.987124,0.998358,0.015,0.974576,0.062921,0.969212
CARDIACX,0.985,0.982906,0.991379,0.987124,0.999795,0.015,0.974576,0.036366,0.969212
Logistic Regression,0.980,0.982759,0.982759,0.982759,0.998666,0.020,0.966102,0.054078,0.958949
KNN,0.960,0.973684,0.956897,0.965217,0.992098,0.040,0.932773,0.120580,0.918359


In [48]:
best_model_name = (
    final_comparison["Accuracy"]
    .idxmax()
)

best_model_name

'Random Forest'

In [49]:
final_comparison.loc[
    [best_model_name]
]

,Accuracy,Precision,Recall,F1 Score,AUC,Hamming Loss,Jaccard Score,Log Loss,MCC
Random Forest,0.995,1.0,0.991379,0.995671,0.999436,0.005,0.991379,0.058751,0.989806


In [50]:
for name, model in best_models.items():

    filename = (
        f"../saved_models/"
        f"{name.lower().replace(' ','_')}.pkl"
    )

    joblib.dump(
        model,
        filename
    )

In [51]:
joblib.dump(
    cardiacx,
    "../saved_models/cardiacx.pkl"
)

['../saved_models/cardiacx.pkl']

In [52]:
baseline_results.to_csv("../results/baseline_results.csv", index=False)

grid_metrics.to_csv("../results/grid_results.csv", index=False)

random_metrics.to_csv("../results/random_results.csv", index=False)

optuna_metrics.to_csv("../results/optuna_results.csv", index=False)

final_comparison.to_csv("../results/final_results.csv", index=False)

In [53]:
print("=" * 60)
print("CARDIACX MODEL EXPERIMENTS COMPLETED")
print("=" * 60)

print(f"\nBest Overall Model : {best_model_name}")

print("\nModels Saved Successfully")

print("\nResults Saved Successfully")

CARDIACX MODEL EXPERIMENTS COMPLETED

Best Overall Model : Random Forest

Models Saved Successfully

Results Saved Successfully


In [54]:

import joblib

for name, model in best_models.items():

    filename = (
        f"../saved_models/"
        f"{name.lower().replace(' ','_')}.pkl"
    )

    joblib.dump(
        model,
        filename
    )

print("Best tuned models saved successfully.")


Best tuned models saved successfully.


In [55]:

joblib.dump(
    cardiacx,
    "../saved_models/cardiacx.pkl"
)

print("CARDIACX saved successfully.")


CARDIACX saved successfully.


In [56]:


best_parameters = {}

for name, model in best_models.items():

    best_parameters[name] = model.get_params()

joblib.dump(
    best_parameters,
    "../saved_models/best_hyperparameters.pkl"
)

print("Best hyperparameters saved successfully.")



Best hyperparameters saved successfully.
